# 20 · 注意力机制：从零实现

> **本节属于 Part 8 · 注意力与 Transformer。这是大模型时代的基石。**

RNN 必须**一步步顺序**处理序列，远距离信息要经过很多步才能传到。**注意力 (attention)** 则让每个位置**一步到位**地"看"序列中任意其他位置，按相关性加权汇聚信息。本节我们从零实现**缩放点积注意力**——Transformer 的核心运算。

## 学习目标

- 理解注意力的 **Query / Key / Value** 框架（"软查表"）
- 实现 **scaled dot-product attention**，理解 $1/\sqrt{d}$ 缩放的作用
- 用**因果掩码**实现自回归注意力（不能看未来）
- 可视化注意力权重

## 直觉与原理

把注意力想成一次"软查表"：

- 每个位置发出一个 **Query**（"我想找什么"），每个位置也提供一个 **Key**（"我是什么"）和一个 **Value**（"我携带的信息"）。
- 用 Query 和所有 Key 的**点积**衡量相关性，经 softmax 变成权重，再对 Value 做**加权平均**。

$$\text{Attention}(Q,K,V) = \text{softmax}\!\Big(\frac{QK^\top}{\sqrt{d}}\Big)V$$

那个 $1/\sqrt{d}$ 很关键：维度 $d$ 大时点积会变得很大，使 softmax 进入饱和区、梯度消失；除以 $\sqrt{d}$ 把方差拉回稳定范围。

In [ ]:
import inspect
import numpy as np
import matplotlib.pyplot as plt
from minitorch import Tensor, nn
from minitorch.nn import scaled_dot_product_attention, causal_mask
from minitorch.utils import numerical_gradient, rel_error

print(inspect.getsource(scaled_dot_product_attention))

## 验证梯度

注意力全部由我们的 Tensor 算子（matmul、softmax）搭成，反向自动完成。用数值梯度检查确认。

In [ ]:
np.random.seed(0)
q = np.random.randn(1, 4, 8); k = np.random.randn(1, 4, 8); v = np.random.randn(1, 4, 8)
R = np.random.randn(1, 4, 8)
tq = Tensor(q)
out, _ = scaled_dot_product_attention(tq, Tensor(k), Tensor(v))
(out * Tensor(R)).sum().backward()
g = numerical_gradient(lambda x: float((scaled_dot_product_attention(Tensor(x), Tensor(k), Tensor(v))[0].data * R).sum()), q.copy())
print("注意力对 Q 的梯度 相对误差:", rel_error(tq.grad, g))

## 可视化注意力

构造一个"每个查询都强烈匹配某个键"的例子，看注意力权重是否如预期聚焦。

In [ ]:
# 让 Q 和 K 相同：每个位置最该关注"自己"（点积最大）
np.random.seed(1)
x = np.random.randn(1, 6, 16)
_, attn = scaled_dot_product_attention(Tensor(x), Tensor(x), Tensor(x))
plt.figure(figsize=(4.2, 3.6))
plt.imshow(attn.data[0], cmap="viridis"); plt.colorbar()
plt.xlabel("key position"); plt.ylabel("query position")
plt.title("Self-attention weights (bright = strong)"); plt.tight_layout(); plt.show()
print("对角线最亮 -> 每个位置最关注自己（因为 Q=K，自相关最高）")

## 因果掩码：不能偷看未来

做语言生成时，预测第 $t$ 个 token 只能用到前 $t$ 个（不能看未来）。**因果掩码**在 softmax 前把"未来"位置的分数设成 $-\infty$，使其权重归零。

In [ ]:
L = 6
x = np.random.randn(1, L, 16)
_, attn = scaled_dot_product_attention(Tensor(x), Tensor(x), Tensor(x), mask=causal_mask(L))
plt.figure(figsize=(4.2, 3.6))
plt.imshow(attn.data[0], cmap="viridis"); plt.colorbar()
plt.title("Causal attention (lower-triangular)"); plt.xlabel("key"); plt.ylabel("query")
plt.tight_layout(); plt.show()
print("上三角全为 0 -> 每个位置只能关注自己及更早的位置")

## PyTorch 对照

PyTorch 也提供了 `F.scaled_dot_product_attention`，公式完全一致。

In [ ]:
import torch
q = torch.randn(1, 4, 8); k = torch.randn(1, 4, 8); v = torch.randn(1, 4, 8)
out_t = torch.nn.functional.scaled_dot_product_attention(q, k, v)
out_o, _ = scaled_dot_product_attention(Tensor(q.numpy()), Tensor(k.numpy()), Tensor(v.numpy()))
print("我们的实现 vs PyTorch 相对误差:", rel_error(out_o.data, out_t.numpy()))

## 📦 沉淀进 minitorch

`scaled_dot_product_attention` 与 `causal_mask` 在 `minitorch/nn/attention.py`，由 `tests/test_transformer.py` 守护。

## 小练习

1. **去掉缩放**：把 $1/\sqrt{d}$ 去掉，把 $d$ 调到 256，观察注意力权重是否变得"非黑即白"（softmax 饱和）。
2. **交叉注意力**：让 Q 来自一个序列、K/V 来自另一个序列（而非自注意力），这正是 seq2seq 解码器用的。
3. **手推 softmax 雅可比**：注意力的反向涉及 softmax 求导，回顾 nb09 的 `softmax - onehot`。

## 小结 & 下一站

✅ 我们从零实现了缩放点积注意力，理解了 Q/K/V 与因果掩码，并可视化了注意力权重。

**下一站 → `21_multihead_and_positional`**：把注意力扩展成**多头**（让模型同时从多个角度关注），并加上**位置编码**（因为注意力本身分不清先后顺序）。